# PhysioGraphSleep — Training Pipeline

Single-channel EEG sleep staging with heterogeneous graph neural networks.

**Target**: Sleep-EDF-20 — 20 subjects, 5-class (W/N1/N2/N3/REM)

## Bu notebook'un ilkeleri
- **Tüm dosyalar `physiographsleep/` klasörünün İÇİNDE** üretilir (cache, checkpoints, logs, outputs).
- **WandB** opsiyonel; `WANDB_ENABLED=True` yapıp `WANDB_API_KEY` ortam değişkenini ayarlayın.
- **Live plots** her epoch sonunda otomatik güncellenir (matplotlib).
- **Detaylı tablo**: train/val loss + accuracy + macro-F1 + per-class F1 her epoch için yazdırılır.

## Kullanım
1. Colab → Runtime → Change runtime type → **GPU (T4/L4/A100)**
2. Hücreleri sırayla çalıştırın.


## 1. Setup & Repo Clone

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Channels contain different.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff frequency.*")
warnings.filterwarnings("ignore", message=".*Lowpass cutoff frequency.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"

# === Yapılandırma flagları ===
WANDB_ENABLED = False  # True yapıp aşağıdaki API key cell'ini doldurun
WANDB_PROJECT = "physiographsleep"
WANDB_RUN_NAME = "sleepedf20-baseline"

# Colab /dev/shm genişlet (multi-worker DataLoader için)
if "google.colab" in sys.modules:
    !df -h /dev/shm 2>/dev/null
    !sudo mount -o remount,size=2G /dev/shm 2>/dev/null
    !df -h /dev/shm 2>/dev/null


In [ ]:
# Repo clone (Colab) veya local repo path tespiti
REPO_URL = "https://github.com/0nur0duncu/physiographsleep.git"

if "google.colab" in sys.modules:
    PROJECT_DIR = "/content/physiographsleep"
    if os.path.exists(PROJECT_DIR):
        !cd {PROJECT_DIR} && git pull
        print(f"Repo güncellendi: {PROJECT_DIR}")
    else:
        !git clone {REPO_URL} {PROJECT_DIR}
        print(f"Repo klonlandı: {PROJECT_DIR}")
    PARENT_DIR = "/content"
else:
    # Local: notebook physiographsleep/ içinde, parent'ı sys.path'e ekle
    PROJECT_DIR = os.path.dirname(os.path.abspath("."))
    if os.path.basename(os.getcwd()) != "physiographsleep":
        PROJECT_DIR = os.path.abspath(".")
    PARENT_DIR = os.path.dirname(PROJECT_DIR)
    print(f"Local repo: {PROJECT_DIR}")

if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
os.chdir(PARENT_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Project root: {PROJECT_DIR}")


In [ ]:
# Bağımlılıklar
!pip install -q -r {PROJECT_DIR}/requirements.txt
if WANDB_ENABLED:
    !pip install -q wandb


In [ ]:
import torch
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 2. Configuration

> Tüm yollar `PROJECT_DIR` (= `physiographsleep/` klasörü) altında. Hiçbir şey bu klasörün dışına yazılmaz.

In [ ]:
from physiographsleep.configs import ExperimentConfig
from physiographsleep.utils.reproducibility import set_seed, get_device

config = ExperimentConfig()

# === HER ŞEY PROJECT_DIR ALTINDA ===
config.data.data_dir       = os.path.join(PROJECT_DIR, "dataset", "sleep-edfx")
config.data.cache_dir      = os.path.join(PROJECT_DIR, "dataset", "cache")
config.train.checkpoint_dir = os.path.join(PROJECT_DIR, "checkpoints")
config.train.log_dir        = os.path.join(PROJECT_DIR, "logs")
OUTPUTS_DIR                 = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(config.train.checkpoint_dir, exist_ok=True)
os.makedirs(config.train.log_dir, exist_ok=True)

# === Veri ayarları ===
config.data.num_subjects = 20
config.data.train_subjects = 14
config.data.val_subjects = 3
config.data.batch_size = 64
config.data.num_workers = 4  # Windows'ta sorun çıkarsa 0'a düşürün

# === Curriculum ===
config.train.curriculum.stage_a_epochs = 30
config.train.curriculum.stage_b_epochs = 25
config.train.curriculum.stage_c_epochs = 25
config.train.patience = 15

set_seed(config.seed)
device = get_device("auto")

print("=" * 60)
print("PATHS (hepsi PROJECT_DIR altında)")
print("=" * 60)
print(f"  data_dir:       {config.data.data_dir}")
print(f"  cache_dir:      {config.data.cache_dir}")
print(f"  checkpoint_dir: {config.train.checkpoint_dir}")
print(f"  log_dir:        {config.train.log_dir}")
print(f"  outputs_dir:    {OUTPUTS_DIR}")
print("=" * 60)
print(f"Device:         {device}")
print(f"Subjects:       {config.data.num_subjects} (train={config.data.train_subjects}, val={config.data.val_subjects})")
print(f"Batch size:     {config.data.batch_size}")
print(f"Num workers:    {config.data.num_workers}")
print(f"Curriculum:     A={config.train.curriculum.stage_a_epochs} | B={config.train.curriculum.stage_b_epochs} | C={config.train.curriculum.stage_c_epochs}")
print(f"Patience:       {config.train.patience}")


In [ ]:
# WandB init (opsiyonel)
wandb_run = None
if WANDB_ENABLED:
    import wandb
    # Colab: secrets'tan API key alın veya kullanıcı interaktif login olur
    try:
        from google.colab import userdata  # type: ignore
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass  # Local: WANDB_API_KEY env veya wandb login

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=OUTPUTS_DIR,  # wandb dosyaları PROJECT_DIR/outputs altında kalır
        config={
            "num_subjects": config.data.num_subjects,
            "batch_size": config.data.batch_size,
            "seq_len": config.data.seq_len,
            "stage_a_epochs": config.train.curriculum.stage_a_epochs,
            "stage_b_epochs": config.train.curriculum.stage_b_epochs,
            "stage_c_epochs": config.train.curriculum.stage_c_epochs,
            "patience": config.train.patience,
        },
    )
    print(f"WandB run: {wandb_run.url}")
else:
    print("WandB devre dışı (WANDB_ENABLED=False)")


## 3. Veri Yükleme + Sınıf Dağılımı

In [ ]:
from physiographsleep.data.loader import load_sleep_edf

data = load_sleep_edf(config.data)

STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
print("Split sizes:")
for split_name, split_data in data.items():
    epochs = split_data["epochs"]
    labels = split_data["labels"]
    spectral = split_data.get("spectral")
    print(f"  {split_name:5s}: {len(labels):6d} epochs | shape={epochs.shape} | spectral={spectral.shape if spectral is not None else 'N/A'}")

print("\nTrain class distribution:")
train_labels = data["train"]["labels"]
counts = np.bincount(train_labels, minlength=5)
total = len(train_labels)
for i, name in enumerate(STAGE_NAMES):
    pct = 100 * counts[i] / total
    bar = "█" * int(pct / 2)
    print(f"  {name:3s}: {counts[i]:6d} ({pct:5.1f}%) {bar}")


## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from physiographsleep.data.dataset import SleepEDFDataset
from physiographsleep.data.transforms import SleepTransforms

train_transform = SleepTransforms(config.data)

train_ds = SleepEDFDataset(
    data["train"]["epochs"], data["train"]["labels"],
    config=config.data, transform=train_transform,
    spectral=data["train"].get("spectral"),
)
val_ds = SleepEDFDataset(
    data["val"]["epochs"], data["val"]["labels"],
    config=config.data,
    spectral=data["val"].get("spectral"),
)

val_loader = DataLoader(
    val_ds, batch_size=config.data.batch_size,
    shuffle=False, num_workers=config.data.num_workers,
    pin_memory=True,
    persistent_workers=config.data.num_workers > 0,
    prefetch_factor=2 if config.data.num_workers > 0 else None,
)

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")


## 5. Model + Sanity Check

In [ ]:
from physiographsleep.models.physiographsleep import PhysioGraphSleep

model = PhysioGraphSleep(config.model)
param_counts = model.count_parameters()

print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name:20s}: {count:>10,}")

# Sanity forward
batch = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    sig = batch["signal"][:2]
    spec = batch["spectral"][:2] if "spectral" in batch else torch.zeros(2, config.data.seq_len, 5, 42)
    out = model(sig, spec)
print(f"\nSanity forward: signal={tuple(sig.shape)} → stage={tuple(out['stage'].shape)} OK")


## 6. Training — Live Plots + WandB + Detailed Epoch Table

Bu hücre `Trainer.callback` hook'unu kullanarak her epoch sonunda:
- Detaylı tablo satırı yazdırır (train loss/acc/MF1, val acc/MF1/κ, per-class F1)
- 4 panelli live plot çizer (loss, accuracy, macro-F1, per-class val F1)
- WandB'ye metrikleri loglar (etkinse)

In [ ]:
import gc
import json
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from physiographsleep.models.losses import MultiTaskLoss
from physiographsleep.data.spectral import SpectralFeatureExtractor
from physiographsleep.training.trainer import Trainer
from physiographsleep.utils.logging_utils import setup_logger

# Free GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, {torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# Class-balanced inverse-frequency weights
class_counts = np.bincount(data["train"]["labels"], minlength=5)
class_weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
class_weights = class_weights / class_weights.sum() * 5
class_weights_tensor = torch.from_numpy(class_weights).float()
print(f"Class weights: {[f'{w:.3f}' for w in class_weights_tensor.tolist()]}")

loss_fn = MultiTaskLoss(config.train.loss, class_weights=class_weights_tensor)
spectral_ext = SpectralFeatureExtractor(config.data)
logger = setup_logger(log_dir=config.train.log_dir)


In [ ]:
# ============================================================
# Training history + per-epoch callback
# ============================================================
HISTORY = {"stage": [], "epoch": [], "train_loss": [],
           "train_acc": [], "train_mf1": [],
           "val_acc": [], "val_mf1": [], "val_kappa": [],
           "val_f1_W": [], "val_f1_N1": [], "val_f1_N2": [],
           "val_f1_N3": [], "val_f1_REM": []}

# Print header once
HEADER_PRINTED = {"v": False}
def _print_header():
    if HEADER_PRINTED["v"]:
        return
    cols = ("Stage", "Epoch", "TrLoss", "TrAcc", "TrMF1",
            "VlAcc", "VlMF1", "Kappa",
            "F1_W", "F1_N1", "F1_N2", "F1_N3", "F1_REM")
    print(" | ".join(f"{c:>7s}" for c in cols))
    print("-" * (10 * len(cols)))
    HEADER_PRINTED["v"] = True

def _draw_live_plot():
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    epochs_x = list(range(len(HISTORY["epoch"])))
    stage_colors = {"A": "#1f77b4", "B": "#2ca02c", "C": "#d62728"}
    stage_marks = [stage_colors.get(s, "k") for s in HISTORY["stage"]]

    # Loss
    axes[0, 0].plot(epochs_x, HISTORY["train_loss"], "-o", ms=3, label="train")
    axes[0, 0].set_title("Loss"); axes[0, 0].set_xlabel("global epoch"); axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend()

    # Accuracy
    axes[0, 1].plot(epochs_x, HISTORY["train_acc"], "-o", ms=3, label="train")
    axes[0, 1].plot(epochs_x, HISTORY["val_acc"],   "-s", ms=3, label="val")
    axes[0, 1].set_title("Accuracy"); axes[0, 1].set_ylim(0, 1); axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()

    # Macro-F1
    axes[1, 0].plot(epochs_x, HISTORY["train_mf1"], "-o", ms=3, label="train")
    axes[1, 0].plot(epochs_x, HISTORY["val_mf1"],   "-s", ms=3, label="val")
    axes[1, 0].set_title("Macro-F1"); axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()

    # Per-class val F1
    for cls in ["W", "N1", "N2", "N3", "REM"]:
        axes[1, 1].plot(epochs_x, HISTORY[f"val_f1_{cls}"], "-o", ms=3, label=cls)
    axes[1, 1].set_title("Val per-class F1"); axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend(ncol=5, fontsize=8)

    # Stage backgrounds
    for ax in axes.flat:
        for i, c in enumerate(stage_marks):
            ax.axvspan(i - 0.4, i + 0.4, alpha=0.04, color=c)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "training_curves.png"), dpi=80, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


def epoch_callback(stage, epoch, train_loss, train_metrics, val_metrics):
    HISTORY["stage"].append(stage)
    HISTORY["epoch"].append(epoch)
    HISTORY["train_loss"].append(train_loss)
    HISTORY["train_acc"].append(train_metrics.get("accuracy", 0.0))
    HISTORY["train_mf1"].append(train_metrics.get("macro_f1", 0.0))
    HISTORY["val_acc"].append(val_metrics.get("accuracy", 0.0))
    HISTORY["val_mf1"].append(val_metrics.get("macro_f1", 0.0))
    HISTORY["val_kappa"].append(val_metrics.get("kappa", 0.0))
    for cls in ["W", "N1", "N2", "N3", "REM"]:
        HISTORY[f"val_f1_{cls}"].append(val_metrics.get(f"f1_{cls}", 0.0))

    _print_header()
    row = (
        f"{stage:>7s}", f"{epoch:>7d}",
        f"{train_loss:>7.4f}",
        f"{train_metrics.get('accuracy', 0):>7.4f}",
        f"{train_metrics.get('macro_f1', 0):>7.4f}",
        f"{val_metrics.get('accuracy', 0):>7.4f}",
        f"{val_metrics.get('macro_f1', 0):>7.4f}",
        f"{val_metrics.get('kappa', 0):>7.4f}",
        f"{val_metrics.get('f1_W', 0):>7.4f}",
        f"{val_metrics.get('f1_N1', 0):>7.4f}",
        f"{val_metrics.get('f1_N2', 0):>7.4f}",
        f"{val_metrics.get('f1_N3', 0):>7.4f}",
        f"{val_metrics.get('f1_REM', 0):>7.4f}",
    )
    print(" | ".join(row))

    # WandB
    if wandb_run is not None:
        log = {
            f"train/loss": train_loss,
            f"train/acc": train_metrics.get("accuracy", 0),
            f"train/macro_f1": train_metrics.get("macro_f1", 0),
            f"val/acc": val_metrics.get("accuracy", 0),
            f"val/macro_f1": val_metrics.get("macro_f1", 0),
            f"val/kappa": val_metrics.get("kappa", 0),
            f"stage": ord(stage) - ord("A"),
        }
        for cls in ["W", "N1", "N2", "N3", "REM"]:
            log[f"val/f1_{cls}"] = val_metrics.get(f"f1_{cls}", 0)
        wandb_run.log(log)

    # Live plot every epoch
    _draw_live_plot()

    # Persist history JSON
    with open(os.path.join(OUTPUTS_DIR, "history.json"), "w") as f:
        json.dump(HISTORY, f, indent=2)


trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_dataset=train_ds,
    train_labels=data["train"]["labels"],
    val_loader=val_loader,
    config=config.train,
    data_config=config.data,
    device=device,
    spectral_extractor=spectral_ext,
    callback=epoch_callback,
)
print("Trainer hazır. Sonraki hücre eğitimi başlatır.")


In [ ]:
# Auto-resume from existing stage checkpoints + run training
from physiographsleep.utils.io_utils import load_checkpoint
from physiographsleep.evaluation.metrics import MetricsCalculator

ckpt_dir = config.train.checkpoint_dir
start_stage = "A"
best_metrics = {}

for stage_name, stage_file, next_stage in [
    ("C", "stage-c.pt", None),
    ("B", "stage-b.pt", "C"),
    ("A", "stage-a.pt", "B"),
]:
    ckpt_path = os.path.join(ckpt_dir, stage_file)
    if os.path.exists(ckpt_path):
        ckpt = load_checkpoint(ckpt_path, device)
        model.load_state_dict(ckpt["model"])
        mf1 = ckpt["metrics"]["macro_f1"]
        print(f"Found {stage_file} (val MF1={mf1:.4f})")
        if next_stage:
            start_stage = next_stage
            print(f"→ Stage {next_stage}'den devam")
        else:
            start_stage = None
            best_metrics = ckpt["metrics"]
            print("→ Tüm stage'ler tamam, post-processing'e geç")
        break
else:
    print("Checkpoint yok, Stage A'dan başlıyorum")

if start_stage:
    best_metrics = trainer.train(start_stage=start_stage)
    print("\n" + "=" * 50)
    print("TRAINING COMPLETE")
    print("=" * 50)
    print(MetricsCalculator.format_report(best_metrics))
else:
    print("\nBest model metrics:")
    print(MetricsCalculator.format_report(best_metrics))


## 7. Test Eval + HMM Post-processing + Confusion Matrices

In [ ]:
from physiographsleep.training.evaluator import Evaluator
from physiographsleep.evaluation.metrics import MetricsCalculator
from physiographsleep.evaluation.postprocessing import HMMPostProcessor, LogitBiasOptimizer
from physiographsleep.evaluation.visualization import plot_confusion_matrix

# Best Stage C checkpoint'i yükle
ckpt_path = os.path.join(config.train.checkpoint_dir, "stage-c.pt")
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Stage C ckpt yüklendi (val MF1={ckpt['metrics']['macro_f1']:.4f})")
else:
    print("stage-c.pt yok, mevcut model state kullanılıyor")

evaluator = Evaluator(device)

if "test" in data:
    test_ds = SleepEDFDataset(
        data["test"]["epochs"], data["test"]["labels"],
        config=config.data,
        spectral=data["test"].get("spectral"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    # Test logits
    test_metrics, test_logits, test_labels_arr = evaluator.evaluate(
        model, test_loader, spectral_ext, return_logits=True,
    )

    # Val logits (logit bias optimizasyonu için)
    val_loader_eval = DataLoader(
        val_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )
    _, val_logits, val_labels_arr = evaluator.evaluate(
        model, val_loader_eval, spectral_ext, return_logits=True,
    )

    # Logit bias optimization
    bias_opt = LogitBiasOptimizer(num_classes=5)
    bias_opt.fit(val_logits, val_labels_arr)
    preds_raw    = test_logits.argmax(axis=1)
    preds_biased = bias_opt.apply(test_logits)

    # HMM post-processing
    hmm = HMMPostProcessor(num_classes=5)
    hmm.fit(data["train"]["labels"])
    preds_hmm = hmm.decode(preds_biased)

    calc = MetricsCalculator()
    raw_m    = calc.compute_all(test_labels_arr, preds_raw)
    biased_m = calc.compute_all(test_labels_arr, preds_biased)
    hmm_m    = calc.compute_all(test_labels_arr, preds_hmm)

    print("\n" + "=" * 50)
    print("TEST RESULTS")
    print("=" * 50)
    print("\n[1] Raw argmax:")
    print(MetricsCalculator.format_report(raw_m))
    print(f"\n[2] + Logit bias (MF1: {raw_m['macro_f1']:.4f} → {biased_m['macro_f1']:.4f}, {biased_m['macro_f1']-raw_m['macro_f1']:+.4f}):")
    print(MetricsCalculator.format_report(biased_m))
    print(f"\n[3] + Logit bias + HMM (MF1: {biased_m['macro_f1']:.4f} → {hmm_m['macro_f1']:.4f}, {hmm_m['macro_f1']-biased_m['macro_f1']:+.4f}):")
    print(MetricsCalculator.format_report(hmm_m))

    # Confusion matrices → PROJECT_DIR/outputs
    for name, preds, m in [("raw", preds_raw, raw_m), ("biased", preds_biased, biased_m), ("hmm", preds_hmm, hmm_m)]:
        cm = MetricsCalculator.confusion_matrix(test_labels_arr, preds)
        plot_confusion_matrix(
            cm,
            save_path=os.path.join(OUTPUTS_DIR, f"cm_{name}.png"),
            title=f"{name} (MF1={m['macro_f1']:.3f})",
        )
    print(f"\nConfusion matrices kaydedildi: {OUTPUTS_DIR}")

    # Final model + WandB summary
    final_path = os.path.join(OUTPUTS_DIR, "final_model.pt")
    torch.save({
        "model": model.state_dict(),
        "config": {"num_subjects": config.data.num_subjects, "batch_size": config.data.batch_size,
                   "seq_len": config.data.seq_len},
        "metrics": {"raw": raw_m, "biased": biased_m, "hmm": hmm_m},
        "param_counts": param_counts,
    }, final_path)
    print(f"Final model: {final_path}  ({param_counts['total']:,} params)")

    if wandb_run is not None:
        wandb_run.summary["test_raw_mf1"]    = raw_m["macro_f1"]
        wandb_run.summary["test_biased_mf1"] = biased_m["macro_f1"]
        wandb_run.summary["test_hmm_mf1"]    = hmm_m["macro_f1"]
        wandb_run.summary["test_hmm_acc"]    = hmm_m["accuracy"]
        for cls in ["W", "N1", "N2", "N3", "REM"]:
            wandb_run.summary[f"test_hmm_f1_{cls}"] = hmm_m.get(f"f1_{cls}", 0)
        wandb_run.save(os.path.join(OUTPUTS_DIR, "*.png"))
        wandb_run.save(os.path.join(OUTPUTS_DIR, "history.json"))
        wandb_run.finish()
        print("WandB run kapatıldı")
else:
    print("Test split yok")
